# 17 Exercises

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

**17-1.** Piecewise-linear models are sometimes an alternative to the nonlinear models described in
[Chapter 18](../18/18.md), replacing a smooth curve by a series of straight-line segments. This exercise deals
with the model shown in Figure 18-4.

(a) Reformulate the model of Figure 18-4 so that it approximates each nonlinear term

```ampl
Trans[i,j] / (1 - Trans[i,j]/limit[i,j])
```

by a piecewise-linear term having three pieces. Set the breakpoints at (1/3) \* limit[i,j]
and (2/3) * limit[i,j]. Pick the slopes so that the approximation equals the original nonlinear
term when Trans[i,j] is 0, 1/3 \* limit[i,j], 2/3 \* limit[i,j], or 11/12 \*
limit[i,j]; you should find that the three slopes are 3/2, 9/2 and 36 in every term, regardless
of the size of limit[i,j]. Finally, place an explicit upper limit of 0.99 \* limit[i,j] on
Trans[i,j].
(b) Solve the approximation with the data given in Figure 18-5, and compare the optimal shipment
amounts to the amounts recommended by the nonlinear model.

\(c) Formulate a more sophisticated version in which the number of linear pieces for each term is
given by a parameter nsl. Pick the breakpoints to be at (k/nsl) * limit[i,j] for k from 1
to nsl-1. Pick the slopes so that the piecewise-linear function equals the original nonlinear function
when Trans[i,j] is (k/nsl) * limit[i,j] for any k from 0 to nsl-1, or when
Trans[i,j] is (nsl-1/4)/nsl * limit[i,j].

Check your model by showing that you get the same results as in (b) when nsl is 3. Then, by trying
higher values of nsl, determine how many linear pieces the approximation requires in order to
determine all shipment amounts to within about 10% of the amounts recommended by the original
nonlinear model.

**17-2.** This exercise asks how you might convert the demand constraints in the transportation
model of [Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a) into the kind of "soft" constraints described in [Section 17.2](./17_2_common_twopiece_and_threepiece_terms.ipynb).
Suppose that instead of a single parameter called demand[j] at each destination j, you are given
the following four parameters that describe a more complicated situation:

```ampl
dem_min_abs[j]       absolute minimum that must be shipped to j
dem_min_ask[j]       preferred minimum amount shipped to j
dem_max_ask[j]       preferred maximum amount shipped to j
dem_max_abs[j]       absolute maximum that may be shipped to j
```

There are also two penalty costs for shipment amounts outside of the preferred limits:

```ampl
dem_min_pen          penalty per unit that shipments fall below dem_min_ask[j]
dem_max_pen          penalty per unit that shipments exceed dem_max_ask[j]
```

Because the total shipped to `j` is no longer fixed, a new variable Receive[j] is introduced to
represent the amount received at j.

(a) Modify the model of [Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a) to use this new information. The modifications will involve
declaring Receive[j] with the appropriate lower and upper bounds, adding a three-piece
piecewise-linear penalty term to the objective function, and substituting Receive[j] for
demand[j] in the constraints.

(b) Add the following demand information to the data of [Figure 3-1b](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1b):

```ampl
dem_min_abs  dem_min_ask  dem_max_ask  dem_max_abs
FRA      800          850          950         1100
DET      975         1100         1225         1250
LAN      600          600          625          625
WIN      350          375          450          500
STL     1200         1500         1800         2000
FRE     1100         1100         1100         1125
LAF      800          900         1050         1175
```

Let dem_min_pen and dem_max_pen be 2 and 4, respectively. Find the new optimal solution.
In the solution, which destinations receive shipments that are outside the preferred levels?

**17-3.** When the diet model of [Figure 2-1](../02/2_2_an_AMPL_model_for_the_diet_problem.ipynb#fig-2-1) is run with the data of [Figure 2-3](../02/2_4_generalizations_to_blending_economics_and_scheduling.ipynb#fig-2-3), there is no feasible
solution. This exercise asks you to use the ideas of [Section 17.2](./17_2_common_twopiece_and_threepiece_terms.ipynb) to find some good near-feasible
solutions.
(a) Modify the model so that it is possible, at a very high penalty, to purchase more than the specified
maximum of a food. In the resulting solution, which maximums are exceeded?

(b) Modify the model so that it is possible, at a very high penalty, to supply more than the specified
maximum of a nutrient. In the resulting solution, which maximums are exceeded?

\(c) Using extremely large penalties, such as 10 20 may give the solver numerical difficulties.
Experiment to see how available solvers behave when you use penalty terms like 10 20 and 10 30 .

**17-4.** In the model of Exercise 4-4(b), the change in crews from one period to the next is limited
to some number M. As an alternative to imposing this limit, suppose that we introduce a new variable
D `t` that represents the change in number of crews (in all shifts) at period t. This variable may
be positive, indicating an increase in crews over the previous period, or negative, indicating a
decrease in crews.

To make use of this variable, we introduce a defining constraint,

```ampl
Dt =      Σ s ∈S   (Y st − Y s,t − 1 ),
```

for each `t` = 1, . . . , T. We then estimate costs of c + per crew added from period to period, and c −
per crew dropped from period to period; as a result, the following cost must be included in the
objective for each month t:

```ampl
c− Dt ,        if D t < 0;
+
c Dt ,         if D t > 0.
```

Reformulate the model in AMPL accordingly, using a piecewise-linear function to represent this
extra cost.

Solve using c − = -20000 and c + = 100000, together with the data previously given. How does this
solution compare to the one from Exercise 4-4(b)?

**17-5.** The following "credit scoring" problem appears in many contexts, including the processing
of credit card applications. A set APPL of people apply for a card, each answering a set QUES
of questions on the application. The response of person `i` to question `j` is converted to a number,
ans[i,j]; typical numbers are years at current address, monthly income, and a home ownership
indicator (say, 1 if a home is owned and 0 otherwise).

To summarize the responses, the card issuer chooses weights Wt[j], from which a score for each
person `i` in APPL is computed by the linear formula

```ampl
sum {j in QUES} ans[i,j] * Wt[j]
```

The issuer also chooses a cutoff, Cut; credit is granted when an applicant's score is greater than or
equal to the cutoff, and is denied when the score is less than the cutoff. In this way the decision
can be made objectively (if not always most wisely).

To choose the weights and the cutoff, the card issuer collects a sample of previously accepted
applications, some from people who turned out to be good customers, and some from people who
never paid their bills. If we denote these two collections of people by sets GOOD and BAD, then the
ideal weights and cutoff (for this data) would satisfy

```ampl
sum {j in QUES} ans[i,j] * Wt[j] >= Cut for each i in GOOD
sum {j in QUES} ans[i,j] * Wt[j] < Cut  for each i in BAD
```

Since the relationship between answers to an application and creditworthiness is imprecise at best,
however, no values of Wt[j] and Cut can be found to satisfy all of these inequalities. Instead,
the issuer has to choose values that are merely the best possible, in some sense. There are any
number of ways to make such a choice; here, naturally, we consider an optimization approach.
(a) Suppose that we define a new variable Diff[i] that equals the difference between person i's
score and the cutoff:

```ampl
Diff[i] = sum {j in QUES} ans[i,j] * Wt[j] - Cut
```

Clearly the undesirable cases are where Diff[i] is negative for `i` in GOOD, and where it is nonnegative
for `i` in BAD. To discourage these cases, we can tell the issuer to minimize the function

```ampl
sum {i in GOOD} max(0,-Diff[i]) + sum {i in BAD} max(0,Diff[i])
```

Explain why minimizing this function tends to produce a desirable choice of weights and cutoff.

(b) The expression above is a piecewise-linear function of the variables Diff[i]. Rewrite it
using AMPL's notation for piecewise-linear functions.

\(c) Incorporate the expression from (b) into an AMPL model for finding the weights and cutoff.

(d) Given this approach, any positive value for Cut is as good as any other. We can `fix` it at a convenient
round number — say, 100. Explain why this is the case.

(e) Using a Cut of 100, apply the model to the following imaginary credit data:

```ampl
set GOOD := _17 _18 _19 _22 _24 _26 _28 _29 ;
set BAD := _15 _16 _20 _21 _23 _25 _27 _30 ;
set QUES := Q1 Q2 R1 R2 R3 S2 T4 ;
param ans:  Q1   Q2   R1   R2   R3   S2   T4 :=
    _15    1.0   10   15   20   10    8   10
    _16    0.0    5   15   40    8   10    8
    _17    0.5   10   25   35    8   10   10
    _18    1.5   10   25   30    8    6   10
    _19    1.5    5   20   25    8    8    8
    _20    1.0    5    5   30    8    8    6
    _21    1.0   10   20   30    8   10   10
    _22    0.5   10   25   40    8    8   10
    _23    0.5   10   25   25    8    8   14
    _24    1.0   10   15   40    8   10   10
    _25    0.0    5   15   15   10   12   10
    _26    0.5   10   15   20    8   10   10
    _27    1.0    5   10   25   10    8    6
    _28    0.0    5   15   40    8   10    8
    _29    1.0    5   15   40    8    8   10
    _30    1.5    5   20   25   10   10   14 ;
```

What are the chosen weights? Using these weights, how many of the good customers would be
denied a card, and how many of the bad risks would be granted one?

You should find that a lot of the bad risks have scores right at the cutoff. Why does this happen in
the solution? How might you adjust the cutoff to deal with it?

(f) To force scores further away from the cutoff (in the desired direction), it might be preferable to
use the following objective,

```ampl
sum {i in GOOD} max(0,-Diff[i]+offset) +
sum {i in BAD} max(0,Diff[i]+offset)
```

where offset is a positive parameter whose value is supplied. Explain why this change has the
desired effect. Try offset values of 2 and 10 and compare the results with those in (e).
(g) Suppose that giving a card to a bad credit risk is considered much more undesirable than refusing
a card to a good credit risk. How would you change the model to take this into account?

(h) Suppose that when someone's application is accepted, his or her score is also used to suggest an
initial credit limit. Thus it is particularly important that bad credit risks not receive very large
scores. How would you add pieces to the piecewise-linear objective function terms to account for
this concern?

**17-6.** In Exercise 18-3, we suggest a way to estimate position, velocity and acceleration values
from imprecise data, by minimizing a nonlinear "sum of squares" function:

```ampl
n
Σ [h j
j=1
                  − (a 0 − a 1 t j − 1⁄2 a 2 t 2j ) ] 2 .
```

An alternative approach instead minimizes a sum of absolute values:

```ampl
n
Σ h j
j=1
                  − (a 0 − a 1 t j − 1⁄2 a 2 t 2j ) .
```

(a) Substitute the sum of absolute values directly for the sum of squares in the model from Exercise 18-3,
first with the abs function, and then with AMPL's explicit piecewise-linear notation.
Explain why neither of these formulations is likely to be handled effectively by any solver.

(b) To model this situation effectively, we introduce variables e `j` to represent the individual formulas
h `j` − (a 0 − a 1 `t` `j` − 1⁄2 a 2 `t` 2j ) whose absolute values are being taken. Then we can express the
minimization of the sum of absolute values as the following constrained optimization problem:

```ampl
n
Minimize               Σ e j 
                       j=1
```

```ampl
Subject to e j = h j − (a 0 − a 1 t j − 1⁄2 a 2 t 2j ), j = 1, . . . , n
```

Write an AMPL model for this formulation, using the piecewise-linear notation for the terms e `j`  .

\(c) Solve for a 0 , a 1 , and a 2 using the data from Exercise 18-3. How much difference is there
between this estimate and the least-squares one?

Use display to print the e `j` values for both the least-squares and the least-absolute-values solutions.
What is the most obvious qualitative difference?

(d) Yet another possibility is to focus on the greatest absolute deviation, rather than the sum:

```ampl
max h j − (a 0 − a 1 t j − 1⁄2 a 2 t 2j ) .
```

```ampl
j = 1 ,. . . ,n
```

Formulate an AMPL linear program that will minimize this quantity, and test it on the same data as
before. Compare the resulting estimates and e `j` values. Which of the three estimates would you
choose in this case?

**17-7.** A planar structure consists of a set of joints connected by bars. For example, in the following
diagram, the joints are represented by circles, and the bars by lines between two circles:

```ampl
1        4
    3
2        5
```

Consider the problem of finding a minimum-weight structure to meet certain external forces. We
let J be the set of joints, and B⊆J×J be the set of admissible bars; for the diagram above, we could
take J = { 1 , 2 , 3 , 4 , 5 }, and

```ampl
B = { ( 1 , 2 ) , ( 1 , 3 ) , ( 1 , 4 ) , ( 2 , 3 ) , ( 2 , 5 ) , ( 3 , 4 ) , ( 3 , 5 ) , ( 4 , 5 ) }.
```

The "origin" and "destination" of a bar are arbitrary. The bar between joints 1 and 2, for example,
could be represented in B by either (1,2) or (2,1), but it need not be represented by both.
We can use two-dimensional Euclidean coordinates to specify the position of each joint in the
plane, taking some arbitrary point as the origin:

```ampl
a xi     horizontal position of joint i relative to the origin
a yi     vertical position of joint i relative to the origin
```

For the example, if the origin lies exactly at joint 2, we might have

```ampl
(a x1 , a y1 ) = (0, 2), (a x2 , a y2 ) = (0, 0), (a x3 , a y3 ) = (2, 1),
(a x4 , a y4 ) = (4, 2), (a x5 , a y5 ) = (4, 0).
```

The remaining data consist of the external forces on the joints:

```ampl
f ix     horizontal component of the external force on joint i
f iy     vertical component of the external force on joint i
```

To resist this force, a subset S⊆J of joints is fixed in position. (It can be proved that fixing two
joints is sufficient to guarantee a solution.)
The external forces induce stresses on the bars, which we can represent as

```ampl
F i j if > 0, tension on bar (i, j)
              if < 0, compression of bar (i, j)
```

A set of stresses is in equilibrium if the external forces, tensions and compressions balance at all
joints, in both the horizontal and vertical components — except at the fixed joints. That is, for
each joint k ∉S,

```ampl
Σ  i ∈J: (i,k) ∈B
                   c xik F ik − Σ
                                   j ∈J: (k, j) ∈B
                                                     c xk j F k j = f kx
Σ                 cy F
    i ∈J: (i,k) ∈B ik ik
                           −   Σ j ∈J: (k, j) ∈B c yk j F k j   = f ky ,
```

where c xst and c yst are the cosines of the direction from joint `s` to joint `t` with the horizontal and vertical
axes,

```ampl
c xst = (a xt − a xs )/ l st ,
c yst = (a yt − a ys )/ l st ,
```

and l st is the length of the bar (s,t):

```ampl
l st = √
       
              (a xt − a xs ) 2 + (a yt − a ys ) 2 .
```

In general, there are infinitely many different sets of equilibrium stresses. However, it can be
shown that a given system of stresses will be realized in a structure of minimum weight if and only
if the cross-sectional areas of the bars are proportional to the absolute values of the stresses. Since
the weight of a bar is proportional to the cross section times length, we can take the (suitably
scaled) weight of bar (i, j) to be

```ampl
w i j = l i j .F i j  .
```

The problem is then to find a system of stresses F `i` `j` that meet the equilibrium conditions, and that
minimize the sum of the weights w `i` `j` over all bars (i, j) ∈ B.

(a) The indexing sets for this linear program can be declared in AMPL as:

```ampl
set joints;
set fixed within joints;
set bars within {i in joints, j in joints: i <> j};
```

Using these set declarations, formulate an AMPL model for the minimum-weight structural design
problem. Use the piecewise-linear notation of this chapter to represent the absolute-value terms in
the objective function.

(b) Now consider in particular a structure that has the following joints:

```ampl
1        2    3    4    5    6
 3.25      7    8    9    10   11
         1.75   12  13    14   15
```

Assume that there is one unit horizontally and vertically between joints, and that the origin is at the
lower left; thus (a x1 ,a y1 ) = (0, 2) and (a x15 ,a y15 ) = (5, 0).

Let there be external forces of 3.25 and 1.75 units straight downward on joints 1 and 7, so that
f 1y = -3.25, f 7y = -1.75, and otherwise all f ix = 0 and f iy = 0. Let S = {6,15}. Finally, let the admissible
bars consist of all possible bars that do not go directly through a joint; for example, (1, 2) or (1, 9)
or (1, 13) would be admissible, but not (1, 3) or (1, 12) or (1, 14).

Determine all the data for the problem that is needed by the linear program, and represent it as
AMPL data statements.

\(c) Use AMPL to solve the linear program and to examine the minimum-weight structure that is
determined.

Draw a diagram of the optimal structure, indicating the cross sections of the bars and the nature of
the stresses. If there is zero force on a bar, it has a cross section of zero, and may be left out of
your diagram.

(d) Repeat parts (b) and \(c) for the case in which all possible bars are admissible. Is the resulting
structure different? Is it any lighter?